# Pfennig 2024 Synechocystis model

Reference: Tobias Pfennig, Elena Kullmann, Tomáš Zavřel, Andreas Nakielski, \
Oliver Ebenhöh, Jan Červený, Gábor Bernát, Anna Barbara Matuszyńska  
"Shedding light on blue-green photosynthesis: A wavelength-dependent mathematical model of photosynthesis in Synechocystis sp. PCC 6803"  
PLOS Computational Biology 20.9 (2024): e1012445.  

[doi: 10.1371/journal.pcbi.1012445](https://www.doi.org/10.1371/journal.pcbi.1012445)

In [ ]:
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from mxlmodels import Simulator, data, get_pfennig2024_synechocystis, plot, scan

In [ ]:
LIGHTS = data.pfennig2024.lights()

model = get_pfennig2024_synechocystis()

res = Simulator(model).simulate(10).get_result().unwrap_or_err()
fig, ax = plot.lines(res.fluxes.loc[:, ["vPS2"]])

## Light sources

In [ ]:
def plot_spectrum(ax, s):
    # Wavelength anchor points and their approximate sRGB colors.
    # Positions are normalized over 400-700 nm so spacing is physically correct.
    _WL_MIN, _WL_MAX = 400, 700
    _SPECTRUM = [
        (400, (0.58, 0.00, 1.00)),  # violet limit
        (450, (0.00, 0.00, 1.00)),  # blue
        (500, (0.00, 1.00, 1.00)),  # cyan
        (550, (0.00, 1.00, 0.00)),  # green
        (580, (1.00, 1.00, 0.00)),  # yellow
        (600, (1.00, 0.50, 0.00)),  # orange
        (650, (1.00, 0.00, 0.00)),  # red
        (700, (1.00, 0.00, 0.00)),  # red limit
    ]

    spectral_cmap = mcolors.LinearSegmentedColormap.from_list(
        "visible_spectrum",
        [((wl - _WL_MIN) / (_WL_MAX - _WL_MIN), color) for wl, color in _SPECTRUM],
    )

    wls = s.index.to_numpy(dtype=float)
    vals = s.to_numpy(dtype=float)

    ax.plot(wls, vals, color="k", linewidth=1)

    # Invisible fill_between to obtain the clip path
    polygon = ax.fill_between(wls, vals, alpha=0)

    # Spectral gradient clipped to the area under the curve
    gradient = ax.imshow(
        np.linspace(0, 1, 256).reshape(1, -1),
        cmap=spectral_cmap,
        aspect="auto",
        extent=[wls.min(), wls.max(), 0, vals.max()],
        alpha=0.4,
    )
    gradient.set_clip_path(polygon.get_paths()[0], transform=ax.transData)
    return ax


fig, axs = plt.subplots(1, 4, figsize=(12, 3), layout="constrained")

plot_spectrum(axs[0], LIGHTS["solar"] / LIGHTS["solar"].max()).set(title="Solar")
plot_spectrum(
    axs[1], LIGHTS["fluorescent_lamp"] / LIGHTS["fluorescent_lamp"].max()
).set(title="Fluorescent lamp")
plot_spectrum(axs[2], LIGHTS["cool_white_led"] / LIGHTS["cool_white_led"].max()).set(
    title="Cool white LEd"
)
plot_spectrum(axs[3], LIGHTS["warm_white_led"] / LIGHTS["warm_white_led"].max()).set(
    title="Warm white LED"
)

for ax in axs:
    ax.set(xlabel="Wavelength (nm)", ylabel="Intensity", ylim=(0, 1.05))

## Re-create fig 

In [ ]:
LIGHT_SOURCES = ["solar", "fluorescent_lamp", "cool_white_led", "warm_white_led"]
PFD = np.arange(20, 500 + 40, 40, dtype=float)


def carbon_uptake(light: str) -> pd.Series:
    """Final CCM flux after 10 000 s, per light intensity, under one light source."""
    res = scan.time_course(
        get_pfennig2024_synechocystis(light_spectrum=LIGHTS[light]),
        to_scan=pd.DataFrame({"PPFD": PFD}),
        time_points=np.linspace(0, 10_000, 100),
    )
    return res.fluxes["vCCM"].groupby(level="n").last().set_axis(PFD)


fig4e = pd.DataFrame({light: carbon_uptake(light) for light in LIGHT_SOURCES})
ax = fig4e.plot()